# LeetCode 1493 — Longest Subarray of 1's After Deleting One Element bằng Python

Link đề: https://leetcode.com/problems/longest-subarray-of-1s-after-deleting-one-element/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def longestSubarray(self, nums: list[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Biến đổi đề: "xoá đúng một phần tử rồi tìm đoạn toàn 1 dài nhất" ⇔ "tìm đoạn con chứa **tối đa một số 0**, đáp án = độ dài − 1".
2. Thành thạo **cửa sổ trượt** (sliding window) hai con trỏ với điều kiện "số lượng 0 trong cửa sổ ≤ 1", kể cả biến thể cửa sổ **không co lại**.
3. Viết lời giải **DP tiền tố/hậu tố**: số 1 liên tiếp kết thúc ngay trước `i` và bắt đầu ngay sau `i`.
4. Viết lời giải **đếm cụm** `O(1)` bộ nhớ: ghép cụm 1 trước và sau một số 0 đơn lẻ.
5. Không quên trường hợp **toàn số 1** (vẫn bắt buộc xoá một phần tử ⇒ `n − 1`).

## 2. Đề bài & ví dụ

Cho mảng nhị phân `nums`. Bạn **phải xoá đúng một** phần tử. Trả về độ dài của đoạn con (subarray, liên tiếp) **không rỗng** dài nhất chỉ gồm số 1 trong mảng sau khi xoá. Nếu không có đoạn nào như vậy, trả `0`.

**Ví dụ 1:** `[1,1,0,1]` → `3` (xoá phần tử ở vị trí 2 ⇒ `[1,1,1]`).

**Ví dụ 2:** `[0,1,1,1,0,1,1,0,1]` → `5` (xoá vị trí 4 ⇒ `[0,1,1,1,1,1,0,1]`, đoạn `[1,1,1,1,1]`).

**Ví dụ 3:** `[1,1,1]` → `2` (bắt buộc xoá một phần tử).

**Ràng buộc:** `1 <= n <= 10^5`, `nums[i]` là `0` hoặc `1`.

In [1]:
import random
import time
from itertools import combinations
from typing import List

examples = [([1, 1, 0, 1], 3), ([0, 1, 1, 1, 0, 1, 1, 0, 1], 5), ([1, 1, 1], 2)]
for nums, want in examples:
    print(nums, "->", want)

[1, 1, 0, 1] -> 3
[0, 1, 1, 1, 0, 1, 1, 0, 1] -> 5
[1, 1, 1] -> 2


## 3. Kiến thức nền

### 3.1 Đoạn con (subarray) và cắt lát

Đoạn con là dãy **liên tiếp** `nums[l..r]`, trong Python là `nums[l:r+1]`. Xoá phần tử `i`: `nums[:i] + nums[i+1:]`.

In [2]:
nums = [0, 1, 1, 1, 0, 1, 1, 0, 1]
print("nums[1:4] =", nums[1:4])
i = 4
print("xoá vị trí 4:", nums[:i] + nums[i + 1:])

nums[1:4] = [1, 1, 1]
xoá vị trí 4: [0, 1, 1, 1, 1, 1, 0, 1]


### 3.2 Đếm chuỗi 1 liên tiếp dài nhất

Duyệt một lượt: gặp 1 thì `cur += 1`, gặp 0 thì `cur = 0`; cập nhật `best = max(best, cur)`.

In [3]:
def longest_ones(a):
    best = cur = 0
    for x in a:
        cur = cur + 1 if x == 1 else 0
        best = max(best, cur)
    return best

print(longest_ones([1, 1, 0, 1, 1, 1, 0]), longest_ones([0, 0]), longest_ones([]))

3 0 0


### 3.3 Cửa sổ trượt hai con trỏ

Giữ cửa sổ `[l, r]`. Mỗi bước mở rộng `r`; nếu cửa sổ vi phạm điều kiện thì tăng `l` cho tới khi hợp lệ. Mỗi con trỏ chỉ đi tới, nên tổng cộng `O(n)`.

In [4]:
# Ví dụ: cửa sổ dài nhất có tổng <= 5 với mảng số dương
a = [2, 1, 3, 1, 1, 4]
l = s = best = 0
for r, x in enumerate(a):
    s += x
    while s > 5:
        s -= a[l]; l += 1
    best = max(best, r - l + 1)
    print(f"r={r} cửa sổ={a[l:r+1]} tổng={s}")
print("dài nhất:", best)

r=0 cửa sổ=[2] tổng=2
r=1 cửa sổ=[2, 1] tổng=3
r=2 cửa sổ=[1, 3] tổng=4
r=3 cửa sổ=[1, 3, 1] tổng=5
r=4 cửa sổ=[3, 1, 1] tổng=5
r=5 cửa sổ=[1, 4] tổng=5
dài nhất: 3


### 3.4 Mảng tiền tố/hậu tố (DP một chiều)

`left[i]` = số 1 liên tiếp **kết thúc tại** `i` (nếu `nums[i] = 0` thì `0`): `left[i] = left[i-1] + 1` nếu `nums[i] = 1`. Tương tự `right[i]` duyệt từ phải.

In [5]:
nums = [1, 1, 0, 1, 1, 1]
n = len(nums)
left = [0] * n
for i in range(n):
    left[i] = (left[i - 1] + 1 if i > 0 else 1) if nums[i] else 0
right = [0] * n
for i in range(n - 1, -1, -1):
    right[i] = (right[i + 1] + 1 if i < n - 1 else 1) if nums[i] else 0
print("left :", left)
print("right:", right)

left : [1, 2, 0, 1, 2, 3]
right: [2, 1, 0, 3, 2, 1]


### 3.5 `itertools.groupby` để gom cụm

`groupby(nums)` trả các cụm giá trị bằng nhau liên tiếp: `(giá trị, iterator)`. Hữu ích để xem cấu trúc "cụm 1 – cụm 0 – cụm 1".

In [6]:
from itertools import groupby
nums = [0, 1, 1, 1, 0, 1, 1, 0, 0, 1]
print([(k, len(list(g))) for k, g in groupby(nums)])

[(0, 1), (1, 3), (0, 1), (1, 2), (0, 2), (1, 1)]


## 4. Ý tưởng nền tảng / trực giác

Làm tay **Ví dụ 2** `[0,1,1,1,0,1,1,0,1]`. Các cụm: `0 | 111 | 0 | 11 | 0 | 1`.

- Xoá một số 0 **đứng một mình giữa hai cụm 1** ⇒ hai cụm nối lại:
  - xoá 0 ở vị trí 4: `3 + 2 = 5`;
  - xoá 0 ở vị trí 7: `2 + 1 = 3`;
  - xoá 0 ở vị trí 0: chỉ được `0 + 3 = 3`.
- Xoá một số 1 thì chỉ làm ngắn đi ⇒ không có lợi (trừ khi mảng **không có số 0**).
- Đáp án `5`.

Nhìn theo đoạn con ban đầu: đoạn `nums[1..6] = [1,1,1,0,1,1]` chứa **đúng một** số 0; xoá nó đi được `6 − 1 = 5`. Tổng quát: **đáp án = (độ dài đoạn dài nhất chứa ≤ 1 số 0) − 1**. Nếu đoạn đó không có số 0 (mảng toàn 1), ta vẫn phải xoá một số 1 ⇒ vẫn là "độ dài − 1". Công thức thống nhất!

In [7]:
nums = examples[1][0]
best = 0
for l in range(len(nums)):
    for r in range(l, len(nums)):
        if nums[l:r + 1].count(0) <= 1:
            best = max(best, r - l + 1)
print("đoạn dài nhất chứa <= 1 số 0 dài", best, "=> đáp án", best - 1)

đoạn dài nhất chứa <= 1 số 0 dài 6 => đáp án 5


## 5. Cách 1 — Brute Force / baseline: thử xoá từng phần tử

**Ý tưởng:** đúng như đề — với mỗi `i`, tạo mảng đã xoá `nums[i]`, tính chuỗi 1 dài nhất (mục 3.2), lấy max.

In [8]:
def longest_brute(nums):
    best = 0
    for i in range(len(nums)):
        rest = nums[:i] + nums[i + 1:]
        best = max(best, longest_ones(rest))
    return best

for nums, want in examples:
    print(nums, "->", longest_brute(nums), "| mong đợi", want)

[1, 1, 0, 1] -> 3 | mong đợi 3
[0, 1, 1, 1, 0, 1, 1, 0, 1] -> 5 | mong đợi 5
[1, 1, 1] -> 2 | mong đợi 2


**Độ phức tạp:** `n` lần xoá × `O(n)` mỗi lần (tạo mảng + quét) ⇒ **`O(n²)`** thời gian, `O(n)` bộ nhớ. Với `n = 10^5` là `10^10` — quá chậm.

**Nút thắt:** mỗi lần xoá ta quét lại **toàn bộ** mảng, trong khi chỉ phần quanh vị trí xoá thay đổi. Hoặc nhìn theo mục 4: chỉ cần tìm đoạn dài nhất chứa ≤ 1 số 0 — bài cửa sổ trượt kinh điển.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (đổi đề):** đáp án = `max(độ dài đoạn chứa ≤ 1 số 0) − 1`.

**Insight 2 (cửa sổ trượt):** điều kiện "≤ 1 số 0" có tính **đơn điệu**: nếu `[l, r]` hợp lệ thì mọi đoạn con bên trong cũng hợp lệ; nếu `[l, r]` vi phạm thì mở rộng thêm vẫn vi phạm ⇒ dùng hai con trỏ.

**Invariant:** sau mỗi bước, cửa sổ `[l, r]` chứa **tối đa một** số 0, và `l` là vị trí **nhỏ nhất** thoả điều đó với `r` hiện tại.

**Quy tắc:**
- Thêm `nums[r]`; nếu là 0 thì `zeros += 1`.
- Trong khi `zeros > 1`: nếu `nums[l] == 0` thì `zeros -= 1`; `l += 1`.
- Cập nhật `best = max(best, r − l)` (độ dài `r − l + 1` trừ 1 phần tử bị xoá).

**Insight 3 (DP):** nếu xoá vị trí `i`, đoạn toàn 1 dài nhất đi qua chỗ xoá có độ dài `left[i−1] + right[i+1]` (số 1 liên tiếp ngay trước và ngay sau `i`). Đáp án = `max_i` của biểu thức này. Xoá vị trí 1 thì chỉ có lợi khi không còn 0 nào — công thức vẫn đúng.

## 7. Các cách tiếp theo

### Cách 2 — Cửa sổ trượt (tối đa một số 0)

In [9]:
def longest_window(nums):
    l = zeros = best = 0
    for r, x in enumerate(nums):
        if x == 0:
            zeros += 1
        while zeros > 1:
            if nums[l] == 0:
                zeros -= 1
            l += 1
        best = max(best, r - l)        # (r - l + 1) - 1
    return best

for nums, want in examples:
    print(longest_window(nums), want)

3 3
5 5
2 2


Code nộp LeetCode (Cách 2):

In [10]:
class SolutionWindow:
    def longestSubarray(self, nums: list[int]) -> int:
        l = zeros = best = 0
        for r, x in enumerate(nums):
            if x == 0:
                zeros += 1
            while zeros > 1:
                if nums[l] == 0:
                    zeros -= 1
                l += 1
            best = max(best, r - l)
        return best

print([SolutionWindow().longestSubarray(a) for a, _ in examples])

[3, 5, 2]


**Độ phức tạp:** `O(n)` thời gian (`l` và `r` mỗi con trỏ đi tối đa `n` bước), `O(1)` bộ nhớ.

**Biến thể cửa sổ không co lại:** thay `while` bằng `if`. Khi vi phạm, cửa sổ chỉ **trượt** (giữ nguyên độ dài) chứ không co ⇒ độ dài cửa sổ không bao giờ giảm và luôn bằng độ dài hợp lệ lớn nhất từng gặp. Cuối cùng đáp án = `n − l − 1`.

In [11]:
def longest_window_nonshrink(nums):
    l = zeros = 0
    for x in nums:
        zeros += x == 0
        if zeros > 1:                  # trượt, không co
            zeros -= nums[l] == 0
            l += 1
    return len(nums) - l - 1

for nums, want in examples:
    print(longest_window_nonshrink(nums), want)

3 3
5 5
2 2


### Cách 3 — DP tiền tố/hậu tố

1. `left[i]`: số 1 liên tiếp kết thúc tại `i`.
2. `right[i]`: số 1 liên tiếp bắt đầu tại `i`.
3. Xoá vị trí `i` ⇒ ghép `left[i−1]` (0 nếu `i = 0`) và `right[i+1]` (0 nếu `i = n−1`).

In [12]:
def longest_dp(nums):
    n = len(nums)
    left = [0] * (n + 1)          # left[i+1] = số 1 liên tiếp kết thúc tại i
    for i in range(n):
        left[i + 1] = left[i] + 1 if nums[i] else 0
    right = [0] * (n + 1)         # right[i] = số 1 liên tiếp bắt đầu tại i
    for i in range(n - 1, -1, -1):
        right[i] = right[i + 1] + 1 if nums[i] else 0
    return max(left[i] + right[i + 1] for i in range(n))

for nums, want in examples:
    print(longest_dp(nums), want)

3 3
5 5
2 2


Dùng mảng dài `n + 1` (có phần tử đệm) để tránh xử lý riêng biên: `left[i]` (trong công thức) là số 1 kết thúc ở `i − 1`, `right[i + 1]` là số 1 bắt đầu ở `i + 1`.

Code nộp LeetCode (Cách 3):

In [13]:
class SolutionDP:
    def longestSubarray(self, nums: list[int]) -> int:
        n = len(nums)
        left = [0] * (n + 1)
        for i in range(n):
            left[i + 1] = left[i] + 1 if nums[i] else 0
        right = [0] * (n + 1)
        for i in range(n - 1, -1, -1):
            right[i] = right[i + 1] + 1 if nums[i] else 0
        return max(left[i] + right[i + 1] for i in range(n))

print([SolutionDP().longestSubarray(a) for a, _ in examples])

[3, 5, 2]


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

### Cách 4 — Đếm cụm, `O(1)` bộ nhớ

Duyệt và giữ hai biến: `cur` = số 1 liên tiếp hiện tại, `prev` = số 1 của cụm ngay trước **một số 0 đơn**. Gặp 0: `prev = cur` (cụm 1 trước số 0 này), `cur = 0`. Gặp 1: `cur += 1`. Ở mỗi bước, `prev + cur` là độ dài nếu xoá số 0 vừa qua. Hai số 0 liền nhau ⇒ khi gặp số 0 thứ hai, `cur = 0` nên `prev = 0` ⇒ không ghép nhầm qua hai số 0.

Riêng mảng không có số 0: kết quả `prev + cur = n` nhưng phải xoá một ⇒ trả `n − 1`.

In [14]:
def longest_groups(nums):
    prev = cur = best = 0
    for x in nums:
        if x == 1:
            cur += 1
        else:
            prev, cur = cur, 0
        best = max(best, prev + cur)
    return best - 1 if best == len(nums) else best

class SolutionGroups:
    def longestSubarray(self, nums: list[int]) -> int:
        return longest_groups(nums)

for nums, want in examples:
    print(longest_groups(nums), want)

3 3
5 5
2 2


`best == len(nums)` chỉ xảy ra khi mảng toàn 1 (nếu có một số 0, `prev + cur ≤ n − 1`). **Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 2** `[0,1,1,1,0,1,1,0,1]`:

| r | nums[r] | zeros (sau thêm) | co `l`? | l | cửa sổ | r − l | best |
|---|---|---|---|---|---|---|---|
| 0 | 0 | 1 | không | 0 | [0] | 0 | 0 |
| 1 | 1 | 1 | không | 0 | [0,1] | 1 | 1 |
| 2 | 1 | 1 | không | 0 | [0,1,1] | 2 | 2 |
| 3 | 1 | 1 | không | 0 | [0,1,1,1] | 3 | 3 |
| 4 | 0 | 2 | có: bỏ nums[0]=0 | 1 | [1,1,1,0] | 3 | 3 |
| 5 | 1 | 1 | không | 1 | [1,1,1,0,1] | 4 | 4 |
| 6 | 1 | 1 | không | 1 | [1,1,1,0,1,1] | 5 | **5** |
| 7 | 0 | 2 | bỏ 1,1,1,0 | 5 | [1,1,0] | 2 | 5 |
| 8 | 1 | 1 | không | 5 | [1,1,0,1] | 3 | 5 |

In [15]:
def trace_window(nums):
    l = zeros = best = 0
    for r, x in enumerate(nums):
        if x == 0:
            zeros += 1
        moved = []
        while zeros > 1:
            if nums[l] == 0:
                zeros -= 1
            moved.append(nums[l]); l += 1
        best = max(best, r - l)
        print(f"r={r} x={x} zeros={zeros} bỏ ra={moved} l={l} cửa sổ={nums[l:r+1]} r-l={r-l} best={best}")
    return best

print("đáp án:", trace_window(examples[1][0]))

r=0 x=0 zeros=1 bỏ ra=[] l=0 cửa sổ=[0] r-l=0 best=0
r=1 x=1 zeros=1 bỏ ra=[] l=0 cửa sổ=[0, 1] r-l=1 best=1
r=2 x=1 zeros=1 bỏ ra=[] l=0 cửa sổ=[0, 1, 1] r-l=2 best=2
r=3 x=1 zeros=1 bỏ ra=[] l=0 cửa sổ=[0, 1, 1, 1] r-l=3 best=3
r=4 x=0 zeros=1 bỏ ra=[0] l=1 cửa sổ=[1, 1, 1, 0] r-l=3 best=3
r=5 x=1 zeros=1 bỏ ra=[] l=1 cửa sổ=[1, 1, 1, 0, 1] r-l=4 best=4
r=6 x=1 zeros=1 bỏ ra=[] l=1 cửa sổ=[1, 1, 1, 0, 1, 1] r-l=5 best=5
r=7 x=0 zeros=1 bỏ ra=[1, 1, 1, 0] l=5 cửa sổ=[1, 1, 0] r-l=2 best=5
r=8 x=1 zeros=1 bỏ ra=[] l=5 cửa sổ=[1, 1, 0, 1] r-l=3 best=5
đáp án: 5


Trace Cách 4 (đếm cụm) trên cùng input, và trên mảng toàn 1:

In [16]:
def trace_groups(nums):
    prev = cur = best = 0
    for i, x in enumerate(nums):
        if x == 1:
            cur += 1
        else:
            prev, cur = cur, 0
        best = max(best, prev + cur)
        print(f"i={i} x={x} prev={prev} cur={cur} prev+cur={prev+cur} best={best}")
    res = best - 1 if best == len(nums) else best
    print("=> kết quả:", res, "(toàn 1, trừ 1)" if best == len(nums) else "")
    return res

trace_groups(examples[1][0])
print()
trace_groups([1, 1, 1])

i=0 x=0 prev=0 cur=0 prev+cur=0 best=0
i=1 x=1 prev=0 cur=1 prev+cur=1 best=1
i=2 x=1 prev=0 cur=2 prev+cur=2 best=2
i=3 x=1 prev=0 cur=3 prev+cur=3 best=3
i=4 x=0 prev=3 cur=0 prev+cur=3 best=3
i=5 x=1 prev=3 cur=1 prev+cur=4 best=4
i=6 x=1 prev=3 cur=2 prev+cur=5 best=5
i=7 x=0 prev=2 cur=0 prev+cur=2 best=5
i=8 x=1 prev=2 cur=1 prev+cur=3 best=5
=> kết quả: 5 

i=0 x=1 prev=0 cur=1 prev+cur=1 best=1
i=1 x=1 prev=0 cur=2 prev+cur=2 best=2
i=2 x=1 prev=0 cur=3 prev+cur=3 best=3
=> kết quả: 2 (toàn 1, trừ 1)


2

## 9. Code nộp LeetCode

Bản chính: cửa sổ trượt (Cách 2).

In [17]:
class Solution:
    def longestSubarray(self, nums: list[int]) -> int:
        l = 0
        zeros = 0
        best = 0
        for r, x in enumerate(nums):
            if x == 0:
                zeros += 1
            while zeros > 1:           # giữ cửa sổ chứa tối đa một số 0
                if nums[l] == 0:
                    zeros -= 1
                l += 1
            best = max(best, r - l)    # độ dài cửa sổ trừ đi phần tử bị xoá
        return best

for nums, want in examples:
    assert Solution().longestSubarray(nums) == want
print("OK ví dụ đề")

OK ví dụ đề


## 10. Vì sao thuật toán đúng

**Bổ đề (đổi đề).** Gọi `W` = độ dài lớn nhất của đoạn con chứa ≤ 1 số 0. Đáp án bằng `W − 1`.
- *(≥)* Lấy đoạn `[l, r]` dài `W` chứa ≤ 1 số 0. Nếu có một số 0, xoá nó ⇒ còn `W − 1` số 1 liên tiếp. Nếu không có số 0, xoá `nums[l]` (một số 1) ⇒ còn `W − 1` số 1 liên tiếp. (Phần tử bị xoá luôn tồn tại vì `W ≥ 1`.)
- *(≤)* Giả sử sau khi xoá vị trí `i` có `L` số 1 liên tiếp. Trong mảng gốc, đoạn đó cộng thêm vị trí `i` (nếu `i` nằm giữa hoặc sát đoạn) tạo đoạn dài `L + 1` chứa ≤ 1 số 0; nếu `i` ở xa đoạn, thì đoạn dài `L` gồm toàn 1 nằm nguyên trong mảng gốc, mở rộng thêm một phần tử kề bên (luôn có vì `n ≥ L + 1`) vẫn chứa ≤ 1 số 0 ⇒ `W ≥ L + 1`.

**Cửa sổ trượt tìm đúng `W`.** Với mỗi `r`, invariant "l nhỏ nhất sao cho `[l, r]` chứa ≤ 1 số 0" đảm bảo `r − l + 1` là đoạn hợp lệ dài nhất **kết thúc tại `r`**. Vì `l` tối ưu không bao giờ giảm khi `r` tăng (tính đơn điệu), ta chỉ cần tăng `l`. Lấy max trên mọi `r` ⇒ `W`.

## 11. Phân tích Time / Space

Gọi `n = len(nums)` (`≤ 10^5`).

| Cách | Time | Space |
|---|---|---|
| 1. Thử xoá từng phần tử | `O(n²)` | `O(n)` |
| 2. Cửa sổ trượt | `O(n)` (mỗi con trỏ ≤ `n` bước) | `O(1)` |
| 2'. Cửa sổ không co | `O(n)` | `O(1)` |
| 3. DP tiền tố/hậu tố | `O(n)` | `O(n)` |
| 4. Đếm cụm | `O(n)` | `O(1)` |

Không có khác biệt average/worst: mọi cách `O(n)` đều quét đúng một (hoặc hai) lượt.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
ALL = {
    "brute": longest_brute,
    "window": longest_window,
    "nonshrink": longest_window_nonshrink,
    "dp": longest_dp,
    "groups": longest_groups,
    "sub_window": lambda a: SolutionWindow().longestSubarray(a),
    "sub_dp": lambda a: SolutionDP().longestSubarray(a),
    "sub_groups": lambda a: SolutionGroups().longestSubarray(a),
    "submit": lambda a: Solution().longestSubarray(a),
}
edge = [
    ([1], 0), ([0], 0),                 # một phần tử: xoá xong mảng rỗng
    ([0, 0, 0], 0),                     # toàn 0
    ([1, 1, 1, 1, 1], 4),               # toàn 1: phải xoá một số 1
    ([1, 0], 1), ([0, 1], 1),
    ([1, 0, 0, 1], 1),                  # hai số 0 liền: không ghép được
    ([1, 1, 0, 0, 1, 1, 1, 0, 1], 4),
    ([0, 1, 1, 1, 1], 4),               # số 0 ở đầu
    ([1, 1, 1, 1, 0], 4),               # số 0 ở cuối
]
for nums, want in edge:
    assert longest_brute(nums) == want, (nums, longest_brute(nums))
for name, f in ALL.items():
    for nums, want in examples + edge:
        assert f(nums) == want, (name, nums, f(nums), want)
print("Mọi cách đúng với", len(examples + edge), "test (edge đã kiểm chứng bằng brute force)")

Mọi cách đúng với 13 test (edge đã kiểm chứng bằng brute force)


### 12.2 Ngẫu nhiên, đối chiếu brute force (seed cố định)

Trộn nhiều mật độ số 1 (từ thưa tới dày) để gặp đủ dạng: toàn 1, toàn 0, nhiều số 0 liền nhau.

In [19]:
rng = random.Random(1493)
for _ in range(3000):
    n = rng.randint(1, 25)
    p = rng.choice([0.1, 0.5, 0.8, 0.95, 1.0])
    nums = [1 if rng.random() < p else 0 for _ in range(n)]
    want = longest_brute(nums)
    for name, f in ALL.items():
        assert f(nums) == want, (name, nums, f(nums), want)
print("3000 mảng ngẫu nhiên: mọi cách khớp brute force")

3000 mảng ngẫu nhiên: mọi cách khớp brute force


### 12.3 Giới hạn `n = 10^5`

In [20]:
bigs = {
    "ngẫu nhiên": [rng.randint(0, 1) for _ in range(10**5)],
    "toàn 1": [1] * 10**5,
    "1 thưa số 0": [0 if rng.random() < 0.001 else 1 for _ in range(10**5)],
}
for label, big in bigs.items():
    res = None
    for name in ["window", "nonshrink", "dp", "groups", "submit"]:
        t0 = time.perf_counter(); r = ALL[name](big); dt = time.perf_counter() - t0
        res = r if res is None else res
        assert r == res
        print(f"{label:12} | {name:9}: {r:6}  ({dt*1000:.1f} ms)")

ngẫu nhiên   | window   :     18  (21.7 ms)
ngẫu nhiên   | nonshrink:     18  (12.4 ms)
ngẫu nhiên   | dp       :     18  (30.5 ms)
ngẫu nhiên   | groups   :     18  (8.8 ms)
ngẫu nhiên   | submit   :     18  (21.7 ms)
toàn 1       | window   :  99999  (13.7 ms)
toàn 1       | nonshrink:  99999  (5.2 ms)


toàn 1       | dp       :  99999  (32.2 ms)
toàn 1       | groups   :  99999  (6.8 ms)
toàn 1       | submit   :  99999  (7.6 ms)
1 thưa số 0  | window   :   7275  (11.7 ms)
1 thưa số 0  | nonshrink:   7275  (6.6 ms)
1 thưa số 0  | dp       :   7275  (25.1 ms)
1 thưa số 0  | groups   :   7275  (6.6 ms)
1 thưa số 0  | submit   :   7275  (11.6 ms)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên trừ phần tử bị xoá

Trả về độ dài cửa sổ `r − l + 1` thay vì `r − l`.

In [21]:
def wrong1(nums):
    l = zeros = best = 0
    for r, x in enumerate(nums):
        zeros += x == 0
        while zeros > 1:
            zeros -= nums[l] == 0
            l += 1
        best = max(best, r - l + 1)          # SAI: chưa trừ phần tử bị xoá
    return best

for nums in [[1, 1, 0, 1], [1, 1, 1], [0]]:
    print(nums, "| sai:", wrong1(nums), "| đúng:", Solution().longestSubarray(nums))

[1, 1, 0, 1] | sai: 4 | đúng: 3
[1, 1, 1] | sai: 3 | đúng: 2
[0] | sai: 1 | đúng: 0


### Lỗi 2 — Chỉ trừ 1 khi cửa sổ có số 0 ⇒ sai với mảng toàn 1

Nghĩ "đáp án = số 1 trong cửa sổ" (tức là `r − l + 1 − zeros`). Với mảng toàn 1, quên rằng **vẫn phải xoá** một phần tử.

In [22]:
def wrong2(nums):
    l = zeros = best = 0
    for r, x in enumerate(nums):
        zeros += x == 0
        while zeros > 1:
            zeros -= nums[l] == 0
            l += 1
        best = max(best, r - l + 1 - zeros)  # SAI khi không có số 0
    return best

for nums in [[1, 1, 1], [1], [1, 1, 0, 1]]:
    print(nums, "| sai:", wrong2(nums), "| đúng:", Solution().longestSubarray(nums))

[1, 1, 1] | sai: 3 | đúng: 2
[1] | sai: 1 | đúng: 0
[1, 1, 0, 1] | sai: 3 | đúng: 3


`[1,1,1]` ⇒ bản sai trả 3 (đúng 2); `[1]` ⇒ trả 1 (đúng 0). **Sửa:** luôn trừ 1 (công thức `r − l`), hoặc xử lý riêng ca toàn 1.

### Lỗi 3 — Ghép hai cụm 1 qua **nhiều** số 0

Ở cách đếm cụm, lưu "cụm 1 trước đó" mà không reset khi gặp số 0 thứ hai liên tiếp.

In [23]:
def wrong3(nums):
    prev = cur = best = 0
    for x in nums:
        if x == 1:
            cur += 1
        elif cur > 0:                        # SAI: chỉ cập nhật prev khi cur > 0
            prev, cur = cur, 0
        best = max(best, prev + cur)
    return best - 1 if best == len(nums) else best

for nums in [[1, 0, 0, 1], [1, 1, 0, 0, 0, 1, 1]]:
    print(nums, "| sai:", wrong3(nums), "| đúng:", Solution().longestSubarray(nums))

[1, 0, 0, 1] | sai: 2 | đúng: 1
[1, 1, 0, 0, 0, 1, 1] | sai: 4 | đúng: 2


`[1,0,0,1]`: bản sai giữ `prev = 1` qua hai số 0 ⇒ ghép `1 + 1 = 2`, nhưng xoá một số 0 vẫn còn số 0 kia ngăn cách ⇒ đúng là 1. **Sửa:** gặp số 0 nào cũng đặt `prev = cur` (cụm ngay trước **chính số 0 đó**, có thể bằng 0).

### Lỗi 4 — Quên giảm `zeros` khi đẩy `l` qua một số 0

Khi co cửa sổ chỉ tăng `l` mà không cập nhật bộ đếm ⇒ `zeros` không bao giờ giảm ⇒ cửa sổ co tới tận `r`.

In [24]:
def wrong4(nums):
    l = zeros = best = 0
    for r, x in enumerate(nums):
        zeros += x == 0
        while zeros > 1 and l <= r:          # (thêm l <= r để không chạy vô hạn)
            l += 1                           # SAI: quên zeros -= 1 khi nums[l] == 0
        best = max(best, r - l)
    return best

for nums in [[0, 1, 1, 1, 0, 1, 1, 0, 1], [1, 0, 1, 0, 1, 1, 1]]:
    print(nums, "| sai:", wrong4(nums), "| đúng:", Solution().longestSubarray(nums))

[0, 1, 1, 1, 0, 1, 1, 0, 1] | sai: 3 | đúng: 5
[1, 0, 1, 0, 1, 1, 1] | sai: 2 | đúng: 4


**Sửa:** `if nums[l] == 0: zeros -= 1` **trước** khi `l += 1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đoạn con dài nhất **sau khi xoá/đổi tối đa k phần tử**" ⇒ cửa sổ trượt với bộ đếm "phần tử xấu ≤ k".
- "Bắt buộc xoá đúng một" ⇒ nhớ trừ 1 **kể cả khi không có phần tử xấu**.
- Mảng nhị phân + cụm ⇒ có thể giải `O(1)` bằng cách nhớ độ dài cụm trước/sau.
- Muốn tránh `while`: cửa sổ không co (`if`), đáp án `n − l − (số phải xoá)`.
- Test bắt buộc: toàn 1, toàn 0, `[1]`, `[0]`, hai số 0 liền nhau, số 0 ở hai đầu.

## 15. Biến thể của bài

1. **LC 1004 — Max Consecutive Ones III:** được **lật** tối đa `k` số 0 ⇒ cửa sổ `zeros ≤ k`, đáp án = độ dài cửa sổ (không trừ).
2. **LC 487 — Max Consecutive Ones II (Premium):** lật tối đa 1 số 0.
3. **Xoá tối đa `k` phần tử** (được phép không xoá): đáp án = số 1 trong cửa sổ chứa ≤ `k` số 0 (Bài 5).
4. **LC 424 — Longest Repeating Character Replacement:** cửa sổ với "số ký tự khác ký tự phổ biến nhất ≤ k".
5. **Nhiều truy vấn trên đoạn `[L, R]`:** cần tiền xử lý độ dài cụm + cấu trúc truy vấn đoạn (sparse table) — khó hơn nhiều.

In [25]:
# Minh hoạ biến thể 1 (LC 1004): lật tối đa k số 0
def max_ones_flip(nums, k):
    l = zeros = best = 0
    for r, x in enumerate(nums):
        zeros += x == 0
        while zeros > k:
            zeros -= nums[l] == 0
            l += 1
        best = max(best, r - l + 1)
    return best

print(max_ones_flip([1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0], 2), max_ones_flip([0, 0, 1, 1], 0))

6 2


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử xoá từng phần tử | Mô phỏng đúng đề | `O(n²)` | `O(n)` | Kiểm chứng, `n` nhỏ |
| 2. Cửa sổ trượt | Đoạn dài nhất chứa ≤ 1 số 0, trừ 1 | `O(n)` | `O(1)` | Bản nộp chuẩn, mở rộng cho `k` |
| 2'. Cửa sổ không co | `if` thay `while`, đáp án `n − l − 1` | `O(n)` | `O(1)` | Code ngắn, ít nhánh |
| 3. DP tiền tố/hậu tố | `left[i−1] + right[i+1]` | `O(n)` | `O(n)` | Dễ chứng minh; cần xoá đúng vị trí cụ thể |
| 4. Đếm cụm | Ghép cụm trước/sau một số 0 | `O(n)` | `O(1)` | Mảng nhị phân, muốn một lượt gọn |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().longestSubarray([1, 0, 1, 1, 0, 1, 1, 1])` bằng bao nhiêu?

### Bài 2 (tính tay)
Với `[1, 1, 0, 1, 1, 1, 0, 1]`, tính mảng `left` (độ dài `n+1`, Cách 3), `right`, và giá trị `left[i] + right[i+1]` cho từng `i`.

### Bài 3 (điền TODO)
Hoàn thiện `max_ones_flip_k(nums, k)` (LC 1004) bằng cửa sổ **không co**.

### Bài 4 (tự viết lại)
Viết lại Cách 3 nhưng trả về thêm **vị trí nên xoá** (một vị trí bất kỳ đạt tối ưu): `best_delete(nums) -> (độ dài, i)`.

### Bài 5 (mini challenge)
Được xoá **tối đa `k`** phần tử (có thể không xoá phần tử nào). Viết `longest_after_delete_k(nums, k)`.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def max_ones_flip_k(nums, k):
    # TODO: cửa sổ không co: khi zeros > k thì trượt l đúng một bước
    pass

# Bài 4 — TODO
def best_delete(nums):
    # TODO: dựng left/right như Cách 3, lưu cả chỉ số đạt max
    pass

# Bài 5 — TODO
def longest_after_delete_k(nums, k):
    # TODO: cửa sổ chứa <= k số 0; đáp án = số số 1 trong cửa sổ
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Các cụm: `1 | 0 | 11 | 0 | 111`. (2) Xoá 0 thứ nhất: `1 + 2 = 3`; xoá 0 thứ hai: `2 + 3 = 5`. (3) Đáp án 5.

**Bài 2.** (1) `left = [0,1,2,0,1,2,3,0,1]`. (2) `right = [2,1,0,3,2,1,0,1,0]`. (3) Tổng theo `i`: `i=2` ⇒ `2 + 3 = 5`; `i=6` ⇒ `3 + 1 = 4`; đáp án 5.

**Bài 3.** (1) `zeros += x == 0`. (2) `if zeros > k: zeros -= nums[l] == 0; l += 1`. (3) Trả `n − l`.

**Bài 4.** (1) Duyệt `i`, tính `v = left[i] + right[i+1]`. (2) Nếu `v > best` thì lưu `(v, i)`. (3) Kiểm tra: xoá `nums[i]` rồi `longest_ones` phải bằng `v`.

**Bài 5.** (1) Cửa sổ `zeros ≤ k`. (2) Giá trị ứng viên `(r − l + 1) − zeros` (số 1 còn lại sau khi xoá hết số 0 trong cửa sổ). (3) Kiểm chứng bằng thử mọi tổ hợp vị trí xoá với `n` nhỏ.

## 19. Đáp án tham khảo

In [27]:
# Bài 1
a1 = Solution().longestSubarray([1, 0, 1, 1, 0, 1, 1, 1])
print(a1); assert a1 == 5

# Bài 2
nums = [1, 1, 0, 1, 1, 1, 0, 1]
n = len(nums)
left = [0] * (n + 1)
for i in range(n):
    left[i + 1] = left[i] + 1 if nums[i] else 0
right = [0] * (n + 1)
for i in range(n - 1, -1, -1):
    right[i] = right[i + 1] + 1 if nums[i] else 0
sums = [left[i] + right[i + 1] for i in range(n)]
print("left =", left, "| right =", right, "| tổng =", sums)
assert left == [0, 1, 2, 0, 1, 2, 3, 0, 1] and right == [2, 1, 0, 3, 2, 1, 0, 1, 0] and max(sums) == 5

5
left = [0, 1, 2, 0, 1, 2, 3, 0, 1] | right = [2, 1, 0, 3, 2, 1, 0, 1, 0] | tổng = [1, 1, 5, 2, 2, 2, 4, 0]


In [28]:
# Bài 3
def max_ones_flip_k_sol(nums, k):
    l = zeros = 0
    for x in nums:
        zeros += x == 0
        if zeros > k:
            zeros -= nums[l] == 0
            l += 1
    return len(nums) - l

# Bài 4
def best_delete_sol(nums):
    n = len(nums)
    left = [0] * (n + 1)
    for i in range(n):
        left[i + 1] = left[i] + 1 if nums[i] else 0
    right = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        right[i] = right[i + 1] + 1 if nums[i] else 0
    best, pos = -1, -1
    for i in range(n):
        v = left[i] + right[i + 1]
        if v > best:
            best, pos = v, i
    return best, pos

rng = random.Random(4)
for _ in range(1500):
    nums = [rng.randint(0, 1) for _ in range(rng.randint(1, 20))]
    k = rng.randint(0, 3)
    assert max_ones_flip_k_sol(nums, k) == max_ones_flip(nums, k)
    v, i = best_delete_sol(nums)
    assert v == Solution().longestSubarray(nums) == longest_ones(nums[:i] + nums[i + 1:])
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [29]:
# Bài 5
def longest_after_delete_k_sol(nums, k):
    l = zeros = best = 0
    for r, x in enumerate(nums):
        zeros += x == 0
        while zeros > k:
            zeros -= nums[l] == 0
            l += 1
        best = max(best, r - l + 1 - zeros)
    return best

def brute_delete_k(nums, k):
    n, best = len(nums), 0
    for t in range(0, min(k, n) + 1):
        for dels in combinations(range(n), t):
            s = set(dels)
            best = max(best, longest_ones([nums[i] for i in range(n) if i not in s]))
    return best

print(longest_after_delete_k_sol([1, 0, 1, 0, 0, 1, 1], 2))
for _ in range(600):
    nums = [rng.randint(0, 1) for _ in range(rng.randint(1, 10))]
    k = rng.randint(0, 3)
    assert longest_after_delete_k_sol(nums, k) == brute_delete_k(nums, k), (nums, k)
print("Đáp án Bài 1–5: mọi assert đều qua")

3
Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template cửa sổ "tối đa k phần tử xấu":**

```python
l = bad = best = 0
for r, x in enumerate(nums):
    bad += is_bad(x)
    while bad > k:
        bad -= is_bad(nums[l])
        l += 1
    best = max(best, r - l + 1)      # bài 1493: r - l (k = 1, bắt buộc xoá một)
return best
```

**Checklist trước khi submit:**
- [ ] Đáp án = độ dài cửa sổ **trừ 1** (kể cả khi mảng toàn 1).
- [ ] Khi đẩy `l` qua số 0 thì **giảm** bộ đếm.
- [ ] Không ghép hai cụm 1 qua **hai** số 0 trở lên.
- [ ] Test: `[1]`, `[0]`, `[1,1,1]`, `[0,0,0]`, `[1,0,0,1]`.
- [ ] Đúng chữ ký: `longestSubarray(self, nums: list[int]) -> int`.